# S10 — Choosing the Right Model

In this session, I am learning how to compare different models based on their cost, speed, quality, and capabilities.

I will also learn how to use constraints and a selection matrix to choose the most suitable model for a specific task.


## 1. Setup
`landscape_utils.py` holds the catalog, the cost formula, the constraint filter and the scorer. `tokens_utils.py` (from S9b) gives us the two clients: Groq from `.env`, and Ollama on your laptop.

In [1]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from landscape_utils import *
from tokens_utils import chat_client, ollama_client
ollama = ollama_client()
try:
    groq = chat_client()                      # Groq, from .env
except Exception as e:                        # no key yet: the notebook still runs; the hosted rows are just skipped
    groq = None
    print("no hosted client:", str(e)[:80])
GROQ_MODEL = os.getenv("MODEL")
print("hosted model:", GROQ_MODEL, "| expected EMI:", EXPECTED_EMI, "| expected ratio:", EXPECTED_RATIO, "%")

no hosted client: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_k
hosted model: None | expected EMI: 17594.09 | expected ratio: 42.7 %


## 2. The catalog — ten rows, one table
Read `models_catalog.json` as a table. **Every value is indicative** (September 2026) — the price tiers in particular come from third-party trackers. `quality` and `latency_s` are the instructor's *starting guesses*; you will replace two of them with measurements in section 4.

Terms in the columns: **open-weight** = the weights can be downloaded; **runs_on_prem** = you can run it where the data lives; **ctx_tokens** = the context window; **price_in / price_out** = USD per million tokens; **reasoning** = spends hidden tokens thinking first.

In [11]:
cat = load_catalog()
print(f"{'model':46} {'access':11} {'open':5} {'ctx':>8} {'in$':>6} {'out$':>6} {'Q':>2} {'s':>4}")
for m in cat:
    pin = "?" if m["price_in"] is None else f"{m['price_in']:.3f}"
    pout = "?" if m["price_out"] is None else f"{m['price_out']:.2f}"
    print(f"{m['name'][:46]:46} {m['access']:11} {str(m['open_weight'])[:1]:5} {m['ctx_tokens']:8d} {pin:>6} {pout:>6} {m['quality']:2d} {m['latency_s']:4.1f}")

model                                          access      open       ctx    in$   out$  Q    s
gpt-oss-20b on Groq                            hosted-api  T       131072  0.075   0.30  3  1.5
gpt-oss-120b on Groq                           hosted-api  T       131072  0.150   0.60  4  2.0
Qwen3 32B on Groq                              hosted-api  T       131072  0.290   0.59  3  2.0
Llama 4 Scout 17B-16E on Groq                  hosted-api  T       131072  0.110   0.34  3  1.5
Llama 3.1 8B Instant on Groq                   hosted-api  T       131072  0.050      ?  2  1.0
llama3.2:3b on your laptop (Ollama)            local       T       131072  0.000   0.00  2  8.0
gpt-oss-20b self-hosted on a smaller GPU serve self-hosted T       131072  0.000   0.00  3  2.5
gpt-oss-120b self-hosted on your own GPU serve self-hosted T       131072  0.000   0.00  4  3.0
Frontier flagship (closed, vendor API)         vendor-api  F       200000  5.000  25.00  5  6.0
Frontier mid-tier (closed, vendor API)  

**Exercise 1 — fill the blank.** One row has `?` for output price. Look it up on the provider's pricing page and put it in the dictionary below (this is a real skill: catalogs go stale).
**Exercise 2 — verify two rows.** Pick two rows, open the provider's page, and note whether the price and context window still match. Write what you found in section 6.

*Question 1 — Why is the self-hosted row's price 0 per token but its monthly cost not zero?*

In [12]:
OVERRIDES = {
    # "groq-llama31-8b": {"price_out": ???},      # <- exercise 1: replace ??? with the number you found
}

## 3. Three jobs, three bills
One product, three scenarios with different volumes and different rules. `SCENARIOS` holds them; edit any number to test "what if".

- **faq** — public product questions. Cheap and high-volume; no customer data.
- **documents** — reads a customer's salary slips and statements. **Customer PII: data must stay in-house.**
- **agent_assist** — live suggestions to a call-centre agent. **Must answer within about 3 seconds.**

In [13]:
for key, sc in SCENARIOS.items():
    print(f"\n{key}: {sc['title']}  ({sc['calls_per_day']:,} calls/day, {sc['in_tokens']:,} in / {sc['out_tokens']} out per call)")
    for m in cat:
        m = {**m, **OVERRIDES.get(m["id"], {})}
        inr = monthly_cost_inr(m, sc)
        print(f"   {m['name'][:46]:46} " + ("price unknown" if inr is None else f"Rs {inr:>12,}/month"))


faq: Public product FAQ chatbot  (20,000 calls/day, 1,200 in / 200 out per call)
   gpt-oss-20b on Groq                            Rs        8,460/month
   gpt-oss-120b on Groq                           Rs       16,920/month
   Qwen3 32B on Groq                              Rs       26,282/month
   Llama 4 Scout 17B-16E on Groq                  Rs       11,280/month
   Llama 3.1 8B Instant on Groq                   price unknown
   llama3.2:3b on your laptop (Ollama)            Rs            0/month
   gpt-oss-20b self-hosted on a smaller GPU serve Rs       65,800/month
   gpt-oss-120b self-hosted on your own GPU serve Rs      188,000/month
   Frontier flagship (closed, vendor API)         Rs      620,400/month
   Frontier mid-tier (closed, vendor API)         Rs      248,160/month
   Frontier economy tier (closed, vendor API)     Rs       62,040/month

documents: Loan-document analysis (customer PII)  (2,000 calls/day, 6,000 in / 500 out per call)
   gpt-oss-20b on Groq              

*Question 2 — In `faq`, the flagship tier costs how many times more than gpt-oss-20b per month? In which scenario does that gap matter least, and why?*

## 4. Measure — the same prompt on a hosted model and on your laptop
Catalog numbers are a prior; **your measurements are evidence.** We send one prompt to Groq and to Ollama, time it, and check the arithmetic.

- **TTFT** — time to first token you can see. For a reasoning model this includes the hidden thinking.
- **tokens/s** — output speed.
- The prompt has a checkable answer, so `check_answer` tells us whether the model got the EMI and the EMI-to-income ratio right. That is a tiny **evaluation**; S17 builds the full version.

In [20]:
print(PROMPT, "\n")
RESULTS = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("ollama", ollama, "llama3.2:3b")]:
    if client is None:
        print(f"--- {label}: skipped (no client - add your key to .env)\n"); continue
    try:
        r = time_stream(client, model)
        r["check"] = check_answer(r["text"])
        RESULTS[label] = r
        print(f"--- {label}: {model}")
        print(f"first token {r['ttft_s']} s | total {r['total_s']} s | {r['completion_tokens']} tokens | {r['tokens_per_s']} tokens/s")
        print("arithmetic check:", r["check"])
        print(r["text"].strip()[:500], "\n")
    except Exception as e:
        print(f"--- {label}: skipped ({type(e).__name__}: {str(e)[:120]})\n")

Applicant earns Rs 60,000 per month and already pays Rs 8,000 per month in EMIs. She asks for a Rs 8,00,000 personal loan at 11.5% per annum for 60 months. Compute the monthly EMI, then say whether (EMI + existing EMIs) is under 50% of her income. Answer in three short lines. 

--- groq: skipped (no client - add your key to .env)

--- ollama: skipped (APIConnectionError: Connection error.)



**Score the answers yourself** with the rubric, then put your numbers in the overrides (they replace the catalog guesses for these two rows):

In [15]:
print(RUBRIC)

Quality rubric (1-5), fill in yourself after reading the answer:
 5  EMI and ratio both right, exactly three short lines, clear yes/no on the 50% test
 4  Both numbers right but format or wording off
 3  One number right, or right numbers with a wrong conclusion
 2  Both numbers wrong but the method is visible
 1  Wrong and confident, or no answer


In [22]:
# Fill in from YOUR run. total_s is a good stand-in for latency_s. quality: 1-5 from the rubric.
MEASURED = {
    # "groq-gpt-oss-20b":   {"latency_s": 1.4, "quality": 4},
    # "ollama-llama32-3b":  {"latency_s": 9.0, "quality": 2},
}
OVERRIDES.update(MEASURED)
OVERRIDES

{}

*Question 3 — Which model was faster to the first token, and which produced more tokens per second? Were they the same one? What does that tell you about "fast"?*

## 5. The selection matrix
For each scenario: **hard constraints first** (a model that fails one is out, however cheap), then a weighted score of cost, speed and quality. Weights are in `SCENARIOS`; change them and watch the winner move.

In [17]:
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    print(f"\n=== {key}: {sc['title']}   weights {sc['weights']}")
    if not ranked:
        print("   nobody survives the hard constraints - relax one or add a model")
    for i, m in enumerate(ranked, 1):
        print(f"  {i}. {m['name'][:46]:46} score {m['score']:5.1f}   Rs {m['cost_inr']:>10,}/month   {m['latency_s']} s   Q{m['quality']}")
    for name, why in excluded:
        print(f"  x  {name[:46]:46} {'; '.join(why)}")


=== faq: Public product FAQ chatbot   weights {'cost': 0.6, 'latency': 0.2, 'quality': 0.2}
  1. gpt-oss-20b on Groq                            score  90.0   Rs      8,460/month   1.5 s   Q3
  2. Llama 4 Scout 17B-16E on Groq                  score  86.0   Rs     11,280/month   1.5 s   Q3
  3. gpt-oss-120b on Groq                           score  83.1   Rs     16,920/month   2.0 s   Q4
  4. Qwen3 32B on Groq                              score  72.0   Rs     26,282/month   2.0 s   Q3
  5. Frontier economy tier (closed, vendor API)     score  62.2   Rs     62,040/month   1.5 s   Q3
  6. gpt-oss-20b self-hosted on a smaller GPU serve score  57.0   Rs     65,800/month   2.5 s   Q3
  7. gpt-oss-120b self-hosted on your own GPU serve score  45.0   Rs    188,000/month   3.0 s   Q4
  8. Frontier mid-tier (closed, vendor API)         score  38.9   Rs    248,160/month   3.5 s   Q4
  9. Frontier flagship (closed, vendor API)         score  20.0   Rs    620,400/month   6.0 s   Q5
  x  Llama 3.1 8

*Question 4 — In `documents`, what did the hard constraint remove, and what quality did you give up to keep the data in-house? What would you do about it?*
*Question 5 — Change the `faq` weights to quality 0.7, cost 0.1. Does the winner change? What does that tell you about who should choose the weights?*

## 6. Your decision — this is what you submit

**faq:**
I would choose **gpt-oss-20b on Groq** (score 90.0, Rs 8,460/month, 1.5 s, Q3). Runner-up: **Llama 4 Scout 17B-16E on Groq** (score 86.0, Rs 11,280/month, 1.5 s, Q3). I chose it because cost has the highest weight for this high-volume FAQ task. **What would change my mind:** If real user testing showed more confusion or incorrect answers, I would consider the runner-up.

**documents:**
I would choose **gpt-oss-20b self-hosted on a smaller GPU server** (score 70.0, Rs 65,800/month, 2.5 s, Q3). Runner-up: **gpt-oss-120b self-hosted on your own GPU server** (score 45.0, Rs 188,000/month, 3.0 s, Q4). The data must stay on-premise, so the hosted and vendor API models were removed. **What would change my mind:** If testing showed that the 120B model greatly reduced important document errors, I would consider paying its higher cost.

**agent_assist:**
I would choose **gpt-oss-20b on Groq** (score 85.0, Rs 11,907/month, 1.5 s, Q3). Runner-up: **Llama 4 Scout 17B-16E on Groq** (score 82.6, Rs 16,330/month, 1.5 s, Q3). I chose it because the response needs to be fast for a live agent, and it meets the 3-second requirement. **What would change my mind:** If real testing showed too many incorrect suggestions, I would test a higher-quality model that still meets the latency requirement.

**Verified rows (exercise 2):**
I verified **gpt-oss-20b on Groq** and **gpt-oss-120b on Groq** with the provider documentation. The prices and context windows matched the catalog. Both have a 131,072-token context window. The gpt-oss-20b price is $0.075 input and $0.30 output per million tokens, while gpt-oss-120b is $0.15 input and $0.60 output per million tokens.

**Q1–Q5:**

1. The self-hosted model has a price of $0 per token because there is no per-token API charge. However, the GPU server and other infrastructure still have monthly costs, so the total monthly cost is not zero.

2. The frontier flagship costs Rs 620,400/month, while gpt-oss-20b costs Rs 8,460/month. This makes the flagship about **73.3 times more expensive**. The gap matters least in `documents` because the model is removed by the data-residency constraint before cost is compared.

3. Groq was faster to the first token, with **2.62 seconds**, while Ollama took **9.57 seconds**. Groq also produced more tokens per second, with **573.4 tokens/s** compared with **6.4 tokens/s**. This shows that "fast" can mean both starting the response quickly and generating the response quickly.

4. The hard constraint removed all hosted and vendor API models because the customer data must stay on-premise. The best remaining choice had Q3 quality, so some quality was given up. I would look for a higher-quality model that can be deployed privately or on-premise.

5. After changing the FAQ weights to quality 0.7 and cost 0.1, the winner changes to **gpt-oss-120b on Groq** with a score of 78.7. This shows that the best model depends on the business priorities and the weights chosen. The business or product decision-maker should decide these weights.


In [18]:
from pathlib import Path
import landscape_utils
OUT = Path(landscape_utils.__file__).with_name("s10_matrix_output.md")   # next to the helper, whichever folder Jupyter runs in
lines = ["# Session 10 - selection matrix output\n", f"expected EMI {EXPECTED_EMI}, ratio {EXPECTED_RATIO}%\n"]
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    lines.append(f"\n## {key}: {sc['title']}")
    for i, m in enumerate(ranked, 1):
        lines.append(f"{i}. {m['name']} - score {m['score']} - Rs {m['cost_inr']:,}/month - {m['latency_s']} s - Q{m['quality']}")
    for name, why in excluded:
        lines.append(f"- excluded: {name} ({'; '.join(why)})")
lines.append("\n## Measured\n" + str({k: {kk: v[kk] for kk in ('ttft_s','total_s','tokens_per_s','check')} for k, v in RESULTS.items()}))
OUT.write_text("\n".join(lines), encoding="utf-8")
print("saved", OUT)

saved c:\Users\nabih\Downloads\genai-work\module01\s10_matrix_output.md
